# Wage prediction

[Causal ML book](https://causalml-book.org/chapters/CausalML_chap_1.pdf)

[Collab notebook](https://colab.research.google.com/github/CausalAIBook/MetricsMLNotebooks/blob/main/PM1/python-ols-and-lasso-for-wage-prediction.ipynb#scrollTo=qb_bGht76jYb)

In [1]:
import numpy as np
import pandas as pd

from tabulate import tabulate
import textwrap

In [2]:
file = "https://raw.githubusercontent.com/CausalAIBook/MetricsMLNotebooks/main/data/wage2015_subsample_inference.csv"
df = pd.read_csv(file)

In [3]:
df.head()

,wage,lwage,sex,shs,hsg,scl,clg,ad,mw,so,we,ne,exp1,exp2,exp3,exp4,occ,occ2,ind,ind2
0,9.615385,2.263364,1,0,0,0,1,0,0,0,0,1,7.0,0.49,0.343,0.2401,3600.0,11,8370.0,18
1,48.076923,3.872802,0,0,0,0,1,0,0,0,0,1,31.0,9.61,29.791,92.3521,3050.0,10,5070.0,9
2,11.057692,2.403126,0,0,1,0,0,0,0,0,0,1,18.0,3.24,5.832,10.4976,6260.0,19,770.0,4
3,13.942308,2.634928,1,0,0,0,0,1,0,0,0,1,25.0,6.25,15.625,39.0625,420.0,1,6990.0,12
4,28.846154,3.361977,1,0,0,0,1,0,0,0,0,1,22.0,4.84,10.648,23.4256,2015.0,6,9470.0,22


In [4]:
df.describe()

,wage,lwage,sex,shs,hsg,scl,clg,ad,mw,so,we,ne,exp1,exp2,exp3,exp4,occ,occ2,ind,ind2
count,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000,5150.000000
mean,23.410410,2.970787,0.444466,0.023301,0.243883,0.278058,0.317670,0.137087,0.259612,0.296505,0.216117,0.227767,13.760583,3.018925,8.235867,25.118038,5310.737476,11.670874,6629.154951,13.316893
std,21.003016,0.570385,0.496955,0.150872,0.429465,0.448086,0.465616,0.343973,0.438464,0.456761,0.411635,0.419432,10.609465,4.000904,14.488962,53.530225,11874.356080,6.966684,5333.443992,5.701019
min,3.021978,1.105912,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,10.000000,1.000000,370.000000,2.000000
25%,13.461538,2.599837,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,0.250000,0.125000,0.062500,1740.000000,5.000000,4880.000000,9.000000
50%,19.230769,2.956512,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,10.000000,1.000000,1.000000,1.000000,4040.000000,13.000000,7370.000000,14.000000
75%,27.777778,3.324236,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,21.000000,4.410000,9.261000,19.448100,5610.000000,17.000000,8190.000000,18.000000
max,528.845673,6.270697,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,47.000000,22.090000,103.823000,487.968100,100000.000000,22.000000,100000.000000,22.000000


In [5]:
# matching table 1.1 in the example
df[['lwage', 'shs', 'hsg', 'scl', 'clg', 'ad', 'we']].mean()

lwage    2.970787
shs      0.023301
hsg      0.243883
scl      0.278058
clg      0.317670
ad       0.137087
we       0.216117
dtype: float64

In [6]:
# make table 1.1 look nicer
print_df = df[['lwage', 'shs', 'hsg', 'scl', 'clg', 'ad', 'we']].mean()
print(tabulate(print_df.to_frame(name='Sample Mean'), floatfmt='.4f', headers='keys'))

         Sample Mean
-----  -------------
lwage         2.9708
shs           0.0233
hsg           0.2439
scl           0.2781
clg           0.3177
ad            0.1371
we            0.2161


In [7]:
# same table by gender
avg_df = df.groupby('sex')[['lwage', 'shs', 'hsg', 'scl', 'clg', 'ad', 'we']].mean()
# transpose to make it look nice
avg_df = avg_df.T
# replace the headers with 0 = male, 1 = female
avg_df = avg_df.rename(columns={0: 'male', 1: 'female'})
print(tabulate(avg_df, floatfmt='.4f'))

-----  ------  ------
lwage  2.9878  2.9495
shs    0.0318  0.0127
hsg    0.2943  0.1809
scl    0.2733  0.2840
clg    0.2940  0.3473
ad     0.1066  0.1752
we     0.2209  0.2101
-----  ------  ------


In [8]:
diff_df = print_df.diff().loc[1]
# 
diff_df = pd.concat([
    avg_df,
    diff_df.to_frame(name='diff')
], axis=1)
print(tabulate(diff_df, floatfmt='.4f', headers='keys'))

print(f"\nIn this sample, women earn {diff_df.loc['lwage', 'diff']:.1%} less than men.")

KeyError: 1

## Prediction

In [9]:
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.20, random_state=123)

In [15]:
ols_mod_base = smf.ols('lwage ~ sex + exp1 + shs + hsg+ scl + clg + mw + so + we + C(occ2) + C(ind2)', data=train_df)
ols_base_result = ols_mod_base.fit()

ols_base_pred = ols_base_result.predict(train_df)
# ols_base_result.summary()


In [16]:
check_cols = ['sex', 'exp1', 'shs', 'hsg', 'scl', 'clg', 'mw', 'so', 'we']
ols_base_result.summary2().tables[1].loc[check_cols]

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
sex,-0.078350,0.016706,-4.689798,2.824423e-06,-0.111103,-0.045596
exp1,0.008832,0.000732,12.061919,6.083628e-33,0.007397,0.010268
shs,-0.604686,0.058416,-10.351333,8.362449e-25,-0.719214,-0.490158
hsg,-0.525626,0.030192,-17.409569,1.568275e-65,-0.584819,-0.466434
scl,-0.421512,0.028156,-14.970457,2.309411e-49,-0.476713,-0.366310
clg,-0.200314,0.025515,-7.850722,5.250772e-15,-0.250338,-0.150290
mw,-0.022234,0.021433,-1.037365,2.996272e-01,-0.064255,0.019787
so,-0.025711,0.020800,-1.236109,2.164893e-01,-0.066490,0.015068
we,0.022578,0.022337,1.010777,3.121833e-01,-0.021215,0.066371


In [17]:
print_str = (
    f"Women in the sample earn {ols_base_result.params['sex']:.4f} less than males,"
    " controlling for job characteristics."
)
print(textwrap.fill(print_str, 80))

Women in the sample earn -0.0783 less than males, controlling for job
characteristics.


**Aside: partialing out**

In [18]:
y_partial = smf.ols('lwage ~ exp1 + shs + hsg+ scl + clg + mw + so + we + C(occ2) + C(ind2)', data=train_df)
y_partial_result = y_partial.fit()

d_partial = smf.ols('sex ~ exp1 + shs + hsg+ scl + clg + mw + so + we + C(occ2) + C(ind2)', data=train_df)
d_partial_result = d_partial.fit()

In [19]:
reg_dict = {
    'y_tilde': y_partial_result.resid, 
    'd_tilde': d_partial_result.resid
}
base_ols_fwl = smf.ols('y_tilde ~ d_tilde', data=reg_dict)
base_ols_fwl_result = base_ols_fwl.fit()

base_ols_fwl_result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                y_tilde   R-squared:                       0.005
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     22.26
Date:                Tue, 29 Sep 2026   Prob (F-statistic):           2.46e-06
Time:                        21:08:00   Log-Likelihood:                -2728.7
No. Observations:                4120   AIC:                             5461.
Df Residuals:                    4118   BIC:                             5474.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   4.054e-14      0.007   5.54e-12      1.000      -0.014       0.014
d_tilde       -0.0783      0.017     -4.718      0.000      -0.111      -0.046
==============================================================================
Omnibus:                      239.403   Durbin-Watson:                   2.016
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              820.563
Skew:                           0.211   Prob(JB):                    6.56e-179
Kurtosis:                       5.145   Cond. No.                         2.27
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [ ]:
def analysis_variance(mod_fit):
    r_squared = mod_fit.rsquared
    rsquared_adj = mod_fit.rsquared_adj
    mse = mse_base = np.mean(mod_fit.resid**2)
    mse_adj = mod_fit.mse_resid

    return pd.Series({
        'r_squared': r_squared,
        'rsquared_adj': rsquared_adj,
        'mse': mse,
        'mse_adj': mse_adj
    })

analysis_variance(ols_fit)

In [ ]:
ols_base_result.predict(test_df)
